# Tarea 2 — Victor Vargas & Leonardo Zeledon
### Universidad Cenfotec — Aprendizaje Automático
### Scikit-Learn | Sin PySpark

**Datasets:**
- Parte I: Heart Disease Dataset (clasificación binaria)
- Parte II: Mall Customers Dataset (clustering)

In [ ]:
# ── Instalación (Google Colab) ────────────────────────────────────────────────
# !pip install scikit-learn matplotlib seaborn scipy pandas numpy -q

import os, warnings, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew

# ── Sklearn: preprocesamiento ─────────────────────────────────────────────────
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold
)

# ── Sklearn: clasificación ────────────────────────────────────────────────────
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# ── Sklearn: clustering ───────────────────────────────────────────────────────
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from scipy.cluster.hierarchy import dendrogram, linkage

# ── Sklearn: métricas ─────────────────────────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    roc_auc_score, roc_curve, auc,
    confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
SEED = 42
np.random.seed(SEED)
print('Entorno listo.')

## 📋 Instrucciones

Completa todas las celdas marcadas con `# TODO` (31 ítems).

**Reglas:** sklearn para todo | sin PySpark | random_state=42 en todos los modelos | `_true_cluster` solo al final | notebook ejecutado sin errores.

**Entrega:** notebook ejecutado (.ipynb) + informe escrito (.pdf).

---

## Restricciones y Buenas Prácticas

⚠ Restricciones obligatorias
- Todo debe realizarse con Python estándar y Scikit-Learn.
- La columna _true_cluster NO debe usarse durante el clustering (solo validación final).
- El notebook debe ejecutarse sin errores de principio a fin antes de la entrega.
- Cada celda de código debe tener al menos un comentario explicativo.
- No usar CrossValidator para K-Fold — usar cross_val_score directamente.

✅ Buenas prácticas evaluadas
- Usar Pipeline de sklearn para encadenar preprocesamiento y modelado.
- Fijar random_state en todos los modelos para reproducibilidad.
- Documentar cada decisión de preprocesamiento con su impacto cuantificado.
- Nombrar variables de forma descriptiva y consistente en todo el notebook.
- Incluir títulos descriptivos en todos los gráficos.

## 0.2. Generar / cargar los datasets

Si tienes los CSV, descomenta la carga. Si no, el bloque siguiente los genera automáticamente.

URL: https://archive.ics.uci.edu/dataset/45/heart+disease
URL: https://www.kaggle.com/datasets/shwetabh123/mall-customers

In [ ]:
# Carga desde CSV local (mismo directorio que el notebook)
df_heart_raw = pd.read_csv('heart_disease.csv')

# Cargar mall_customers y ajustar nombres de columnas al esquema del notebook
df_mall_raw = (pd.read_csv('mall_customers.csv')
               .rename(columns={'Annual Income (k$)': 'Annual_Income_k',
                                'Spending Score (1-100)': 'Spending_Score'})
               .drop(columns='CustomerID'))

# Crear _true_cluster con KMeans k=5 — referencia para validación final
from sklearn.preprocessing import StandardScaler as _SS
from sklearn.cluster import KMeans as _KM
_X_ref = _SS().fit_transform(df_mall_raw[['Age', 'Annual_Income_k', 'Spending_Score']])
df_mall_raw['_true_cluster'] = _KM(n_clusters=5, random_state=SEED, n_init=10).fit_predict(_X_ref)

print(f'Heart Disease: {df_heart_raw.shape} | target: {df_heart_raw.target.value_counts().to_dict()}')
print(f'Mall Customers: {df_mall_raw.shape}')


---
# PARTE I — CLASIFICACIÓN: HEART DISEASE

## 1. Carga y exploración inicial

In [ ]:
df_heart = df_heart_raw.copy()
print(f'Filas: {df_heart.shape[0]}  Columnas: {df_heart.shape[1]}')
display(df_heart.head(10))
print('\nTipos de datos:')
print(df_heart.dtypes)
print('\nNulos por columna:')
print(df_heart.isnull().sum())

## 2. EDA

In [ ]:
# TODO 2: Estadísticas descriptivas completas
display(df_heart.describe(include='all').T.round(3))
print(f'\nTipos: {df_heart.dtypes.value_counts().to_dict()}  | Nulos: {df_heart.isnull().sum().sum()}')


In [ ]:
# TODO 3: Distribución del target con barras y porcentajes
fig, ax = plt.subplots(figsize=(6, 4))
counts = df_heart['target'].value_counts().sort_index()
labels = ['Sin enfermedad (0)', 'Con enfermedad (1)']
bars = ax.bar(labels, counts.values, color=['steelblue', 'tomato'], edgecolor='black')
for bar, val in zip(bars, counts.values):
    pct = 100 * val / len(df_heart)
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f'{val}\n({pct:.1f}%)', ha='center', fontweight='bold')
ax.set_title('Distribución del Target — Heart Disease', fontsize=13)
ax.set_ylabel('Frecuencia')
plt.tight_layout(); plt.show()


In [ ]:
# TODO 4: Histogramas de todas las variables en grilla
# Variables continuas (usadas en KDE y boxplots posteriores)
num_cols = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']

all_feat = [c for c in df_heart.columns if c != 'target']
ncols = 4
nrows = (len(all_feat) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, nrows * 3))
axes = axes.flatten()
for i, col in enumerate(all_feat):
    axes[i].hist(df_heart[col].dropna(), bins=20, color='steelblue',
                 edgecolor='black', alpha=0.8)
    axes[i].set_title(col, fontsize=10)
for j in range(len(all_feat), len(axes)):
    axes[j].set_visible(False)
plt.suptitle('Histogramas de todas las variables — Heart Disease', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# TODO 5: KDE por clase con media marcada como línea punteada vertical
colors_map = {0: 'steelblue', 1: 'tomato'}
labels_map = {0: 'Sin enfermedad', 1: 'Con enfermedad'}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    for t in [0, 1]:
        subset = df_heart[df_heart['target'] == t][col].dropna()
        sns.kdeplot(subset, ax=axes[i], fill=True, alpha=0.4,
                    color=colors_map[t], label=labels_map[t])
        axes[i].axvline(subset.mean(), color=colors_map[t], linestyle='--',
                        linewidth=1.8, label=f'Media {labels_map[t]} ({subset.mean():.1f})')
    axes[i].set_title(f'KDE — {col}', fontsize=10)
    axes[i].legend(fontsize=7)
axes[-1].set_visible(False)
plt.suptitle('Densidad (KDE) por clase — variables continuas', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# d de Cohen
def cohen_d(df, feature_cols, group_col='target'):
    results = []
    for col in feature_cols:
        g0 = df[df[group_col]==0][col].dropna()
        g1 = df[df[group_col]==1][col].dropna()
        n0, n1 = len(g0), len(g1)
        v0, v1 = g0.var(ddof=1), g1.var(ddof=1)
        std_p = np.sqrt(((n1-1)*v1 + (n0-1)*v0) / (n1+n0-2)) if (v0 and v1) else 1
        d = (g1.mean() - g0.mean()) / std_p if std_p else 0
        abs_d = abs(d)
        efecto = 'Grande' if abs_d>=0.8 else ('Mediano' if abs_d>=0.5 else ('Pequeño' if abs_d>=0.2 else 'Insignificante'))
        results.append({'variable':col,'media_sano':round(g0.mean(),3),
                        'media_enf':round(g1.mean(),3),'d_cohen':round(d,4),'efecto':efecto})
    return pd.DataFrame(results).sort_values('d_cohen', key=abs, ascending=False)

cohen_pd = cohen_d(df_heart, num_cols+['ca','thal','exang','cp','slope'])
print(cohen_pd.to_string(index=False))

colors_d = {'Grande':'tomato','Mediano':'orange','Pequeño':'steelblue','Insignificante':'lightgrey'}
fig, ax = plt.subplots(figsize=(10,5))
ax.barh(cohen_pd['variable'][::-1], cohen_pd['d_cohen'].abs()[::-1],
        color=[colors_d[e] for e in cohen_pd['efecto'][::-1]], edgecolor='black')
for u, ls, lb in [(0.2,'--','pequeño'),(0.5,'-.','mediano'),(0.8,':','grande')]:
    ax.axvline(u, color='black', linestyle=ls, alpha=0.5, label=f'|d|={u} ({lb})')
ax.set_xlabel('d de Cohen (|valor absoluto|)')
ax.set_title('Tamaño del efecto por variable — Heart Disease', fontsize=12)
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

In [ ]:
# TODO 7: Curtosis con .kurtosis() — > 1 leptocúrtica, [-1, 1] mesocúrtica, < -1 platicúrtica
kurt = df_heart[num_cols].kurtosis()
print('Curtosis (exceso de Fisher) por variable:')
for col, k in kurt.items():
    tipo = ('Leptocúrtica (colas pesadas, >1)' if k > 1
            else ('Mesocúrtica (aprox. normal)' if abs(k) <= 1
                  else 'Platicúrtica (colas ligeras, <-1)'))
    print(f'  {col:<12}: {k:>7.3f}  → {tipo}')

fig, ax = plt.subplots(figsize=(8, 4))
bar_colors = ['tomato' if k > 1 else ('gold' if k < -1 else 'steelblue') for k in kurt.values]
ax.barh(kurt.index, kurt.values, color=bar_colors, edgecolor='black')
ax.axvline(0,  color='black', linewidth=0.8)
ax.axvline( 1, color='red',  linestyle='--', alpha=0.6, label='Umbral leptocúrtica (+1)')
ax.axvline(-1, color='gold', linestyle='--', alpha=0.6, label='Umbral platicúrtica (−1)')
ax.set_title('Curtosis de Fisher — variables numéricas', fontsize=12)
ax.set_xlabel('Curtosis'); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()


In [ ]:
# TODO 8: Mapa de calor de correlaciones de Pearson (triángulo inferior)
corr = df_heart.corr(numeric_only=True)
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            mask=mask, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlaciones de Pearson — Heart Disease', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()


In [ ]:
# TODO 9: Boxplots de variables numéricas por clase (target 0 vs 1)
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.boxplot(x='target', y=col, data=df_heart, ax=axes[i],
                palette={0: 'steelblue', 1: 'tomato'})
    axes[i].set_title(f'Boxplot — {col}', fontsize=10)
    axes[i].set_xticklabels(['Sin enfermedad (0)', 'Con enfermedad (1)'], fontsize=8)
    axes[i].set_xlabel('')
axes[-1].set_visible(False)
plt.suptitle('Boxplots por clase — variables continuas', fontsize=13, y=1.01)
plt.tight_layout(); plt.show()


## 3. Tratamiento de valores faltantes

In [ ]:
COLS_MISSING = ['ca', 'thal']

# Estrategia A: imputación con la media (SimpleImputer)
imputer = SimpleImputer(strategy='mean')
df_imputed = df_heart.copy()
df_imputed[COLS_MISSING] = imputer.fit_transform(df_heart[COLS_MISSING])
print(f'Filas tras imputación : {len(df_imputed)} | Nulos restantes: {df_imputed.isnull().sum().sum()}')

# Estrategia B: eliminar filas con nulos
df_dropna = df_heart.dropna(subset=COLS_MISSING)
print(f'Filas tras dropna     : {len(df_dropna)}')
print('\nDecisión: se usa imputación con la media — preserva más datos (303 vs 291).')

## 4. Detección y manejo de outliers (IQR)

In [ ]:
# TODO 12: Método IQR — cuantificar outliers; boxplots antes/después
def iqr_resumen(df, cols):
    rows = {}
    for col in cols:
        Q1, Q3 = df[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        n = int(((df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)).sum())
        rows[col] = {'Q1': round(Q1, 2), 'Q3': round(Q3, 2), 'IQR': round(IQR, 2),
                     'n_outliers': n, 'pct_%': round(100*n/len(df), 2)}
    return pd.DataFrame(rows).T

print('Outliers detectados por IQR (df_imputed):')
print(iqr_resumen(df_imputed, num_cols).sort_values('n_outliers', ascending=False).to_string())

# Winsorización: recortar a límites IQR (no elimina filas)
df_clean = df_imputed.copy()
for col in num_cols:
    Q1, Q3 = df_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df_clean[col] = df_clean[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_imputed[num_cols].boxplot(ax=axes[0]); axes[0].set_title('ANTES de recorte IQR')
axes[0].tick_params(axis='x', rotation=30)
df_clean[num_cols].boxplot(ax=axes[1]);   axes[1].set_title('DESPUÉS de recorte IQR')
axes[1].tick_params(axis='x', rotation=30)
plt.suptitle('Manejo de outliers — winsorización por IQR', fontsize=12)
plt.tight_layout(); plt.show()
print(f'Filas conservadas: {len(df_clean)} (winsorización no elimina filas)')


## 5. Preparación de features y Pipeline

In [ ]:
FEATURE_COLS = [c for c in df_clean.columns if c != 'target']
X = df_clean[FEATURE_COLS].values
y = df_clean['target'].values

# Split estratificado 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y)

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print(f'Distribución train — 0:{(y_train==0).sum()} 1:{(y_train==1).sum()}')
print(f'Distribución test  — 0:{(y_test==0).sum()}  1:{(y_test==1).sum()}')

# Pipeline de preprocesamiento
prep_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler())
])
X_train_s = prep_pipeline.fit_transform(X_train)
X_test_s  = prep_pipeline.transform(X_test)
print('\nPipeline (Imputer → StandardScaler) aplicado correctamente.')

## 6. Modelado — Tres algoritmos de clasificación

In [ ]:
# Función de evaluación reutilizable
def evaluar_modelo(nombre, y_true, y_pred, y_prob=None):
    metrics = {
        'Modelo':    nombre,
        'Accuracy':  round(accuracy_score(y_true, y_pred), 4),
        'F1':        round(f1_score(y_true, y_pred, average='macro'), 4),
        'Precision': round(precision_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'Recall':    round(recall_score(y_true, y_pred, average='macro', zero_division=0), 4),
        'AUC-ROC':   round(roc_auc_score(y_true, y_prob[:,1]) if y_prob is not None else 0.0, 4),
    }
    print(f"\n{nombre}")
    for k,v in metrics.items():
        if k != 'Modelo': print(f'  {k:<12}: {v}')
    return metrics

results_clf = {}

### 6.1 Decision Tree Classifier

In [ ]:
# TODO 15: Decision Tree — probar max_depth=3, 5 y 10; guardar el mejor por AUC-ROC
best_dt, best_dt_auc = None, 0
for depth in [3, 5, 10]:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=SEED)
    dt.fit(X_train_s, y_train)
    y_pred = dt.predict(X_test_s)
    y_prob = dt.predict_proba(X_test_s)
    m = evaluar_modelo(f'DT max_depth={depth}', y_test, y_pred, y_prob)
    if m['AUC-ROC'] > best_dt_auc:
        best_dt_auc = m['AUC-ROC']
        best_dt = dt
        best_dt_metrics = dict(m)

best_dt_metrics['Modelo'] = 'Decision Tree'
results_clf['DT'] = best_dt_metrics
print(f'\nMejor DT seleccionado: max_depth={best_dt.max_depth}  AUC-ROC={best_dt_auc:.4f}')


In [ ]:
# TODO 16: Visualizar el árbol con plot_tree (max_depth=4 para legibilidad)
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(best_dt, max_depth=4,
          feature_names=FEATURE_COLS,
          class_names=['Sin enfermedad', 'Con enfermedad'],
          filled=True, rounded=True, fontsize=7, ax=ax)
plt.title(f'Árbol de Decisión — Heart Disease (max_depth={best_dt.max_depth}, visualizado hasta nivel 4)',
          fontsize=12)
plt.tight_layout(); plt.show()


### 6.2 Random Forest Classifier

In [ ]:
# TODO 17: Random Forest (n_estimators=200) con importancia de variables
rf = RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1)
rf.fit(X_train_s, y_train)
y_pred_rf = rf.predict(X_test_s)
y_prob_rf  = rf.predict_proba(X_test_s)
m_rf = evaluar_modelo('Random Forest', y_test, y_pred_rf, y_prob_rf)
m_rf['Modelo'] = 'Random Forest'
results_clf['RF'] = m_rf

# Importancia de variables (mean decrease impurity)
importancias = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, 6))
importancias.plot(kind='barh', ax=ax, color='steelblue', edgecolor='black', alpha=0.85)
ax.set_title('Importancia de variables — Random Forest (200 árboles)', fontsize=12)
ax.set_xlabel('Importancia (mean decrease impurity)')
plt.tight_layout(); plt.show()


### 6.3 MLP Classifier (sklearn)

In [ ]:
# TODO 18: MLP — arquitecturas (64,32) y (128,64,32); solvers adam y sgd
# adam: convergencia rápida con tasa adaptativa; sgd: requiere más iteraciones y es más sensible al lr
best_mlp, best_mlp_auc = None, 0
for arch in [(64, 32), (128, 64, 32)]:
    for solver in ['adam', 'sgd']:
        mlp = MLPClassifier(hidden_layer_sizes=arch, solver=solver,
                            max_iter=600, random_state=SEED, early_stopping=True)
        mlp.fit(X_train_s, y_train)
        y_pred = mlp.predict(X_test_s)
        y_prob = mlp.predict_proba(X_test_s)
        m = evaluar_modelo(f'MLP {arch} {solver}', y_test, y_pred, y_prob)
        if m['AUC-ROC'] > best_mlp_auc:
            best_mlp_auc = m['AUC-ROC']
            best_mlp = mlp
            best_mlp_metrics = dict(m)

best_mlp_metrics['Modelo'] = 'MLP'
results_clf['MLP'] = best_mlp_metrics
print(f'\nMejor MLP: {best_mlp.hidden_layer_sizes}  solver={best_mlp.solver}  AUC-ROC={best_mlp_auc:.4f}')


## 7. Evaluación y comparación de modelos

In [ ]:
# TODO 19: Tabla comparativa ordenada por AUC-ROC
df_results = (pd.DataFrame([results_clf['DT'], results_clf['RF'], results_clf['MLP']])
              .set_index('Modelo')
              .drop(columns='Modelo', errors='ignore')
              .sort_values('AUC-ROC', ascending=False))
print('Comparativa de modelos (ordenados por AUC-ROC):')
display(df_results.style
        .highlight_max(color='#c8f7c5', axis=0)
        .highlight_min(color='#f7c5c5', axis=0)
        .format('{:.4f}'))


In [ ]:
# TODO 20: Curvas ROC del mejor y peor modelo; punto óptimo de Youden (TPR − FPR máximo)
models_roc = {
    'Decision Tree': (y_test, best_dt.predict_proba(X_test_s)[:, 1]),
    'Random Forest': (y_test, rf.predict_proba(X_test_s)[:, 1]),
    'MLP':           (y_test, best_mlp.predict_proba(X_test_s)[:, 1]),
}
auc_scores = {name: roc_auc_score(yt, yp) for name, (yt, yp) in models_roc.items()}
best_name  = max(auc_scores, key=auc_scores.get)
worst_name = min(auc_scores, key=auc_scores.get)

palette_roc = {'Decision Tree': 'orange', 'Random Forest': 'steelblue', 'MLP': 'seagreen'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: todos superpuestos
for name, (yt, yp) in models_roc.items():
    fpr, tpr, _ = roc_curve(yt, yp)
    axes[0].plot(fpr, tpr, linewidth=2, color=palette_roc[name],
                 label=f'{name} (AUC={auc(fpr, tpr):.3f})')
    j = np.argmax(tpr - fpr)
    axes[0].scatter(fpr[j], tpr[j], marker='*', s=130, color=palette_roc[name], zorder=5)
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set(xlabel='FPR', ylabel='TPR', title='Curvas ROC — todos los modelos')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

# Panel derecho: mejor vs peor con punto de Youden etiquetado
for name in [best_name, worst_name]:
    yt, yp = models_roc[name]
    fpr, tpr, _ = roc_curve(yt, yp)
    j = np.argmax(tpr - fpr)
    auc_val = auc(fpr, tpr)
    axes[1].plot(fpr, tpr, linewidth=2, color=palette_roc[name],
                 label=f'{name} (AUC={auc_val:.3f})')
    axes[1].scatter(fpr[j], tpr[j], marker='*', s=220, color=palette_roc[name], zorder=6,
                    label=f'Youden ({name}): FPR={fpr[j]:.2f}, TPR={tpr[j]:.2f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[1].set(xlabel='FPR', ylabel='TPR', title=f'ROC — {best_name} (mejor) vs {worst_name} (peor)')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

plt.suptitle('Análisis de Curvas ROC — Heart Disease', fontsize=13)
plt.tight_layout(); plt.show()


In [ ]:
# TODO 21: Matriz de confusión del mejor modelo (heatmap seaborn)
best_model = {'Decision Tree': best_dt, 'Random Forest': rf, 'MLP': best_mlp}[best_name]
y_pred_best = best_model.predict(X_test_s)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5, cbar=False,
            xticklabels=['Sin enfermedad', 'Con enfermedad'],
            yticklabels=['Sin enfermedad', 'Con enfermedad'])
ax.set_title(f'Matriz de Confusión — {best_name}', fontsize=13)
ax.set_xlabel('Predicho', fontsize=11); ax.set_ylabel('Real', fontsize=11)
plt.tight_layout(); plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'TP={tp}  FP={fp}  FN={fn}  TN={tn}')
print(f'\n⚠ FN={fn}: pacientes con enfermedad clasificados como sanos — el error más costoso en clínica.')
print(f'  FP={fp}: pacientes sanos clasificados como enfermos — genera pruebas innecesarias.')
print('\n' + classification_report(y_test, y_pred_best,
                                   target_names=['Sin enfermedad', 'Con enfermedad']))


In [ ]:
# TODO 22: Validación cruzada 5-Fold con cross_val_score; F1 promedio ± std
from sklearn.base import clone

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_f1 = cross_val_score(clone(best_model), X_train_s, y_train,
                        cv=skf, scoring='f1_macro', n_jobs=-1)

ci_low  = cv_f1.mean() - 1.96 * cv_f1.std()
ci_high = cv_f1.mean() + 1.96 * cv_f1.std()
print(f'F1 macro por fold: {np.round(cv_f1, 4)}')
print(f'Promedio ± std   : {cv_f1.mean():.4f} ± {cv_f1.std():.4f}')
print(f'IC 95%           : [{ci_low:.4f}, {ci_high:.4f}]')

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(1, 6), cv_f1, color='steelblue', edgecolor='black', alpha=0.85)
ax.axhline(cv_f1.mean(), color='tomato', linestyle='--', linewidth=2,
           label=f'Media: {cv_f1.mean():.3f}')
ax.fill_between([-0.5, 5.5], ci_low, ci_high, alpha=0.15, color='tomato', label='IC 95%')
ax.set(xlabel='Fold', ylabel='F1 macro',
       title=f'Validación cruzada 5-Fold — {best_name}',
       xticks=range(1, 6), ylim=(0.4, 1.05))
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


---
# PARTE II — CLUSTERING: MALL CUSTOMERS

## 8. EDA del dataset de clustering

In [ ]:
df_mall = df_mall_raw.copy()
# _true_cluster se guarda aparte y se elimina del DataFrame de trabajo
true_clusters = df_mall.pop('_true_cluster')
print(f'Dimensiones: {df_mall.shape}')
display(df_mall.head(5))
print('\nNulos:', df_mall.isnull().sum().to_dict())

In [ ]:
SPEND_COLS = ['Age', 'Annual_Income_k', 'Spending_Score']

# Estadísticas descriptivas
display(df_mall[SPEND_COLS].describe().round(3))

# Distribución de Genre
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
genre_vc = df_mall['Genre'].value_counts()
axes[0].bar(genre_vc.index, genre_vc.values, color=['steelblue','tomato'], edgecolor='black')
axes[0].set_title('Distribución por Género')
df_mall[SPEND_COLS].hist(bins=20, ax=axes[1], color='steelblue', edgecolor='black')
plt.suptitle('EDA — Mall Customers', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# TODO 25: Codificar Genre, mapa de calor de correlaciones, skewness
le = LabelEncoder()
df_mall['Genre_enc'] = le.fit_transform(df_mall['Genre'])
print(f'Codificación Genre: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Skewness de variables de clustering
skew_vals = df_mall[SPEND_COLS].skew()
print('\nSkewness:')
for col, s in skew_vals.items():
    interp = 'Fuertemente asimétrica' if abs(s) > 1 else ('Moderadamente asimétrica' if abs(s) > 0.5 else 'Aprox. simétrica')
    print(f'  {col:<20}: {s:>6.3f}  → {interp}')

# Mapa de calor de correlaciones
corr_mall = df_mall[SPEND_COLS + ['Genre_enc']].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr_mall, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Correlaciones — Mall Customers', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()


In [ ]:
# TODO 26: Outliers IQR sobre variables de clustering; winsorización
print('Outliers detectados por IQR (Mall Customers):')
for col in SPEND_COLS:
    Q1, Q3 = df_mall[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    n = int(((df_mall[col] < Q1 - 1.5*IQR) | (df_mall[col] > Q3 + 1.5*IQR)).sum())
    print(f'  {col:<20}: {n:>3} ({100*n/len(df_mall):.1f}%)')

# Winsorización — conserva todas las filas
df_mall_clean = df_mall.copy()
for col in SPEND_COLS:
    Q1, Q3 = df_mall_clean[col].quantile([0.25, 0.75])
    IQR = Q3 - Q1
    df_mall_clean[col] = df_mall_clean[col].clip(lower=Q1 - 1.5*IQR, upper=Q3 + 1.5*IQR)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_mall[SPEND_COLS].boxplot(ax=axes[0]); axes[0].set_title('ANTES de recorte IQR')
df_mall_clean[SPEND_COLS].boxplot(ax=axes[1]); axes[1].set_title('DESPUÉS de recorte IQR')
plt.suptitle('Outliers — Mall Customers (winsorización IQR)', fontsize=12)
plt.tight_layout(); plt.show()
print(f'Filas conservadas: {len(df_mall_clean)}')


## 9. Preparación de features para clustering

In [ ]:
scaler_mall = StandardScaler()
X_mall = scaler_mall.fit_transform(df_mall_clean[SPEND_COLS])
print(f'Features escaladas: {X_mall.shape}')

# Índice de Hopkins
from sklearn.neighbors import NearestNeighbors
from random import sample as rnd_sample

def hopkins_index(X, m=None, seed=SEED):
    n, d = X.shape
    m = m or int(0.1*n)
    nbrs = NearestNeighbors(n_neighbors=1).fit(X)
    rand_idx = rnd_sample(range(n), m)
    rng_h = np.random.default_rng(seed)
    ujd, wjd = [], []
    for j in range(m):
        u = rng_h.uniform(X.min(axis=0), X.max(axis=0), d).reshape(1,-1)
        ujd.append(nbrs.kneighbors(u, 2, return_distance=True)[0][0][1])
        wjd.append(nbrs.kneighbors(X[[rand_idx[j]]], 2, return_distance=True)[0][0][1])
    H = sum(ujd)/(sum(ujd)+sum(wjd))
    return round(float(H), 4)

H = hopkins_index(X_mall)
print(f'Índice de Hopkins: {H} ({"Apto para clustering" if H>0.7 else "Revisar estructura"})')

## 10. Selección de k — Elbow + Silhouette

In [ ]:
K_RANGE = range(2, 11)
inertias, sil_scores = [], []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_mall)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_mall, labels, metric='euclidean'))
    print(f'  k={k:>2}  Inercia={inertias[-1]:>12,.2f}  Silhouette={sil_scores[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(list(K_RANGE), inertias, 'o--', color='steelblue', linewidth=2)
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inercia (WCSS)')
axes[0].set_title('Método del Codo'); axes[0].grid(alpha=0.3)

best_k = list(K_RANGE)[int(np.argmax(sil_scores))]
axes[1].plot(list(K_RANGE), sil_scores, 's--', color='tomato', linewidth=2)
axes[1].axvline(best_k, color='darkred', linestyle=':', label=f'Mejor k={best_k}')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Análisis de Silhouette'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Selección de k — Mall Customers', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()
print(f'k seleccionado: {best_k}')
K_FINAL = best_k

## 11. Tres algoritmos de clustering

In [ ]:
# TODO 29: K-Means, AgglomerativeClustering y GMM con K_FINAL
results_clust = {}

# ── K-Means ───────────────────────────────────────────────────────────────────
km = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=10)
labels_km = km.fit_predict(X_mall)
sil_km = silhouette_score(X_mall, labels_km)
results_clust['K-Means'] = {'Silhouette': sil_km, 'Inercia': km.inertia_}
print(f'K-Means          Silhouette={sil_km:.4f}  Inercia={km.inertia_:.2f}')

# ── Agglomerative — probar ward, complete, average ───────────────────────────
best_agg_sil, best_linkage, labels_agg = -1, '', None
for link in ['ward', 'complete', 'average']:
    lbl = AgglomerativeClustering(n_clusters=K_FINAL, linkage=link).fit_predict(X_mall)
    sil = silhouette_score(X_mall, lbl)
    print(f'Agglomerative ({link:>8})  Silhouette={sil:.4f}')
    if sil > best_agg_sil:
        best_agg_sil, best_linkage, labels_agg = sil, link, lbl
results_clust['Agglomerative'] = {'Silhouette': best_agg_sil, 'Linkage': best_linkage}
print(f'  → Mejor linkage: {best_linkage}')

# Dendrograma (ward sobre todos los datos)
Z = linkage(X_mall, method='ward')
fig, ax = plt.subplots(figsize=(12, 5))
dendrogram(Z, truncate_mode='lastp', p=30, ax=ax,
           above_threshold_color='steelblue', color_threshold=0)
ax.axhline(y=Z[-(K_FINAL), 2], color='tomato', linestyle='--', linewidth=2,
           label=f'Corte sugerido para k={K_FINAL}')
ax.set(title='Dendrograma — Agglomerative Clustering (linkage=ward)',
       xlabel='Muestra (truncado)', ylabel='Distancia')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

# ── GMM — BIC y AIC para selección de componentes ────────────────────────────
k_range_gmm = range(2, 9)
bics, aics = [], []
for k in k_range_gmm:
    g = GaussianMixture(n_components=k, random_state=SEED, n_init=3)
    g.fit(X_mall)
    bics.append(g.bic(X_mall)); aics.append(g.aic(X_mall))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(k_range_gmm), bics, 'o--', color='steelblue', linewidth=2, label='BIC')
ax.plot(list(k_range_gmm), aics, 's--', color='tomato',   linewidth=2, label='AIC')
ax.axvline(K_FINAL, color='black', linestyle=':', linewidth=1.5, label=f'k={K_FINAL}')
ax.set(xlabel='k', ylabel='Criterio de información',
       title='GMM — BIC vs AIC por número de componentes')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

gmm = GaussianMixture(n_components=K_FINAL, random_state=SEED, n_init=5)
labels_gmm = gmm.fit_predict(X_mall)
sil_gmm = silhouette_score(X_mall, labels_gmm)
results_clust['GMM'] = {'Silhouette': sil_gmm}
print(f'GMM              Silhouette={sil_gmm:.4f}')

print('\nResumen Silhouette:')
for alg, res in results_clust.items():
    print(f'  {alg:<15}: {res["Silhouette"]:.4f}')


## 12. Visualización con TSNE (PCA previo)

In [ ]:
# TODO 30: PCA (curva de varianza acumulada) + t-SNE; 3 paneles comparativos
# ── PCA — varianza acumulada ──────────────────────────────────────────────────
pca_full = PCA(n_components=X_mall.shape[1], random_state=SEED)
pca_full.fit(X_mall)
var_cum = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, len(var_cum) + 1), var_cum, 'o--', color='steelblue', linewidth=2)
ax.axhline(0.95, color='tomato', linestyle='--', alpha=0.7, label='95% varianza')
ax.set(xlabel='Número de componentes', ylabel='Varianza acumulada explicada',
       title='PCA — Varianza acumulada', xticks=range(1, len(var_cum) + 1))
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()
n_pca = int(np.argmax(var_cum >= 0.95)) + 1
print(f'Componentes para ≥95% varianza: {n_pca} de {X_mall.shape[1]}')

# ── t-SNE sobre las features escaladas ───────────────────────────────────────
tsne = TSNE(n_components=2, perplexity=30, random_state=SEED)
X_tsne = tsne.fit_transform(X_mall)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
configs = [('K-Means', labels_km, 'tab10'),
           ('Agglomerative', labels_agg, 'Set1'),
           ('GMM', labels_gmm, 'tab20')]
for ax, (title, labels, cmap) in zip(axes, configs):
    sc = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=labels, cmap=cmap,
                    s=45, alpha=0.85, edgecolors='k', linewidths=0.3)
    ax.set_title(f't-SNE — {title}', fontsize=11)
    ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
    plt.colorbar(sc, ax=ax, label='Cluster')
plt.suptitle('Visualización t-SNE coloreada por algoritmo de clustering', fontsize=13, y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# Validación con _true_cluster — adjusted_rand_score compara etiquetas predichas vs referencia
from sklearn.metrics import adjusted_rand_score

print('Adjusted Rand Score vs _true_cluster (1.0 = perfecto, 0.0 = aleatorio):')
for name, labels in [('K-Means', labels_km), ('Agglomerative', labels_agg), ('GMM', labels_gmm)]:
    ari = adjusted_rand_score(true_clusters, labels)
    print(f'  {name:<15}: ARI = {ari:.4f}')

# Visualizar t-SNE coloreado por _true_cluster vs mejor predicho
best_alg_name = max(results_clust, key=lambda k: results_clust[k]['Silhouette'])
best_labels_val = {'K-Means': labels_km, 'Agglomerative': labels_agg, 'GMM': labels_gmm}[best_alg_name]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (title, lbl) in zip(axes, [('_true_cluster (referencia)', true_clusters.values),
                                    (f'{best_alg_name} (predicho)', best_labels_val)]):
    sc = ax.scatter(X_tsne[:, 0], X_tsne[:, 1], c=lbl, cmap='tab10',
                    s=45, alpha=0.85, edgecolors='k', linewidths=0.3)
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
    plt.colorbar(sc, ax=ax, label='Cluster')
plt.suptitle('Comparación: clusters reales vs predichos (t-SNE)', fontsize=13)
plt.tight_layout(); plt.show()


## 13. Perfilado e interpretación de clusters

In [ ]:
# Mejor algoritmo por Silhouette
best_alg = max(results_clust, key=lambda k: results_clust[k]['Silhouette'])
best_labels = {'K-Means': labels_km, 'Agglomerative': labels_agg, 'GMM': labels_gmm}[best_alg]
print(f'Mejor algoritmo: {best_alg}')

df_mall_clean['cluster'] = best_labels

# Perfil en escala original
profile = (df_mall_clean.groupby('cluster')[SPEND_COLS]
           .mean().round(2))
profile['n_clientes'] = df_mall_clean.groupby('cluster').size()
print('\nPerfil de clusters:')
display(profile)

In [ ]:
# Perfilado visual de clusters — heatmap + barras agrupadas
profile = df_mall_clean.groupby('cluster')[SPEND_COLS].mean().round(1)
profile['n_clientes'] = df_mall_clean.groupby('cluster').size()

# Normalizar a [0,1] para el heatmap
profile_norm = (profile[SPEND_COLS] - profile[SPEND_COLS].min()) / \
               (profile[SPEND_COLS].max() - profile[SPEND_COLS].min())

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap con valores originales anotados
sns.heatmap(profile_norm.T, annot=profile[SPEND_COLS].T, fmt='.1f',
            cmap='YlOrRd', ax=axes[0], linewidths=0.5,
            cbar_kws={'label': 'Escala normalizada'})
axes[0].set_title('Perfil de clusters — medias originales (escala de color normalizada)', fontsize=10)
axes[0].set_xlabel('Cluster'); axes[0].set_ylabel('Variable')

# Barras agrupadas
x = np.arange(len(SPEND_COLS))
w = 0.8 / K_FINAL
colors_cl = plt.cm.tab10(np.linspace(0, 1, K_FINAL))
for i, cl in enumerate(profile.index):
    axes[1].bar(x + i * w, profile.loc[cl, SPEND_COLS], w,
                label=f'Cluster {cl} (n={int(profile.loc[cl,"n_clientes"])})',
                color=colors_cl[i], edgecolor='black', alpha=0.85)
axes[1].set_xticks(x + w * (K_FINAL - 1) / 2)
axes[1].set_xticklabels(SPEND_COLS, rotation=15)
axes[1].set_title('Medias por variable y cluster', fontsize=11)
axes[1].set_ylabel('Valor medio (escala original)')
axes[1].legend(title='Cluster', fontsize=8, bbox_to_anchor=(1.01, 1))
plt.tight_layout(); plt.show()

# Interpretación cualitativa
mean_income  = profile['Annual_Income_k'].mean()
mean_spend   = profile['Spending_Score'].mean()
print('\nInterpretación de segmentos:')
for cl in profile.index:
    row = profile.loc[cl]
    ingreso = 'alto' if row['Annual_Income_k'] > mean_income else 'bajo'
    gasto   = 'alto' if row['Spending_Score']   > mean_spend   else 'bajo'
    print(f'  Cluster {cl}: ingreso {ingreso}, gasto {gasto} | '
          f'Edad media={row["Age"]:.0f} | n={int(row["n_clientes"])}')
